# 01 数据理解

目标：在建模和可视化之前，先理解 Olist 电商数据集的 9 张表。

本阶段重点回答：

- 每张表有多少行、多少列？
- 每张表的主键/连接键可能是什么？
- 哪些字段有缺失值？
- 哪些表是订单事实表，哪些表是客户、商品、卖家等维度表？


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


## 1. 定位数据文件

这段代码兼容两种情况：

- Notebook 在项目根目录运行
- Notebook 在 `notebooks/` 目录运行


In [ ]:
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
raw_data_dir = project_root / "data" / "raw"

csv_files = sorted(raw_data_dir.glob("*.csv"))

print(f"Project root: {project_root}")
print(f"Raw data dir: {raw_data_dir}")
print(f"CSV files: {len(csv_files)}")

for file in csv_files:
    print(file.name)


## 2. 读取全部 CSV

这里用文件名作为表名，比如 `olist_orders_dataset.csv` 会变成 `olist_orders_dataset`。


In [ ]:
tables = {}

for file in csv_files:
    table_name = file.stem
    tables[table_name] = pd.read_csv(file)
    print(f"{table_name}: {tables[table_name].shape}")


## 3. 数据规模和质量概览


In [ ]:
summary = []

for name, df in tables.items():
    summary.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "missing_pct": round(df.isna().sum().sum() / df.size * 100, 2),
        "duplicate_rows": int(df.duplicated().sum()),
    })

summary_df = pd.DataFrame(summary).sort_values("rows", ascending=False)
summary_df


## 4. 查看每张表的字段类型和前几行

先从订单表开始，因为它通常是这个项目的中心事实表。


In [ ]:
orders = tables["olist_orders_dataset"]

orders.info()
orders.head()


In [ ]:
for name, df in tables.items():
    print("=" * 100)
    print(name)
    print(df.dtypes)
    display(df.head(3))


## 5. 下一步

跑完这个 Notebook 后，可以整理出第一版数据字典和表关系：

- `orders` 连接 `customers`
- `orders` 连接 `order_items`
- `orders` 连接 `payments`
- `orders` 连接 `reviews`
- `order_items` 连接 `products` 和 `sellers`
- `products` 连接品类英文翻译表
